In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision
import torchvision.transforms as transforms
from torchvision.transforms import autoaugment
from torch.utils.checkpoint import checkpoint
import torch.optim as optim
import copy
import contextlib
import numpy as np

In [2]:
def get_cifar10_loaders(
    batch_size: int         = 256, 
    num_workers: int        = 8,  
    pin_memory: bool        = True,
    persistent_workers: bool = True, 
):
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(p=0.5),
        autoaugment.AutoAugment(autoaugment.AutoAugmentPolicy.CIFAR10),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])

    trainset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform_train
    )
    trainloader = torch.utils.data.DataLoader(
        trainset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        drop_last=True,              
    )

    testset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform_test
    )
    testloader = torch.utils.data.DataLoader(
        testset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
    )

    return trainloader, testloader

In [3]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, channels // reduction, kernel_size=1)
        self.fc2 = nn.Conv2d(channels // reduction, channels, kernel_size=1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        w = self.global_avg_pool(x)
        w = self.relu(self.fc1(w))
        w = 2 * self.sigmoid(self.fc2(w))
        return x * w 

In [4]:
class BottleneckBlock(nn.Module):
    def __init__(self, in_channels, out_channels, max_depth=3, reduction_factor=4):
        super(BottleneckBlock, self).__init__()

        reduced_channels = in_channels // reduction_factor

        self.conv1 = nn.Conv2d(in_channels, reduced_channels, kernel_size=1, bias=False)
        self.bn1 = nn.GroupNorm(1, in_channels)

        self.conv2 = nn.Conv2d(reduced_channels, reduced_channels, kernel_size=3, stride=1, padding=1, bias=False, groups=reduced_channels)
        self.bn2 = nn.GroupNorm(1, reduced_channels)
        
        self.conv3 = nn.Conv2d(reduced_channels, out_channels, kernel_size=1, bias=False)
        self.bn3 = nn.GroupNorm(1, reduced_channels)

        self.relu = nn.ReLU()

        self.skip_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.skip_bn = nn.GroupNorm(1, in_channels)

        self.se = SEBlock(out_channels)

    def forward(self, x, depth):
        identity = x

        out = self.relu(x)
        out = self.bn1(out)
        out = self.conv1(out)

        out = self.relu(out)
        out = self.bn2(out)
        out = self.conv2(out)

        out = self.relu(out)
        out = self.bn3(out)
        out = self.conv3(out)

        out = self.se(out)

        skip = self.relu(identity)
        skip = self.skip_bn(skip)
        skip = self.skip_conv(skip)

        out += skip
        return out

In [5]:
class AmplifyConv(nn.Module):
    def __init__(self, channels, max_depth=3, num_module=2):
        super(AmplifyConv, self).__init__()
        self.max_depth = max_depth
        self.num_module = num_module

        self.conv = nn.ModuleList([
            BottleneckBlock(channels, channels, max_depth) for _ in range(num_module)
        ])
        self.bn = nn.GroupNorm(1, channels)

        self.step_embeddings = nn.Parameter(torch.randn(max_depth, channels, 1, 1))
        self.step_embeddings1 = nn.Parameter(torch.randn(max_depth, channels, 1, 1))
        self.step_embeddings2 = nn.Parameter(torch.randn(max_depth, channels, 1, 1))

    def step_forward(self, x, x_prev, depth):
        out = self.conv[depth // (self.max_depth // self.num_module)](x, depth)
        x = 2 * F.sigmoid(self.step_embeddings1[depth]) * F.relu(x) + (1 + self.step_embeddings[depth]) * out
        x = self.bn(x)

        x = x + 2 * F.sigmoid(self.step_embeddings2[depth]) * F.relu(x_prev)
        return x

    def forward(self, x):
        prev = []
        for d in range(self.max_depth):
            if d % 2 == 0:
                prev.append(x)
                
            x_prev = prev[((d + 1) - ((d + 1) & -(d + 1))) // 2]
            x = checkpoint(self.step_forward, x, x_prev, d, use_reentrant=False)

        return x

In [6]:
class RecursionAmplifyConv(nn.Module):
    def __init__(self, channels, height, width):
        super(RecursionAmplifyConv, self).__init__()
        
        self.conv0 = nn.Conv2d(3, channels, kernel_size=4, stride=2, padding=1)
        self.amconv = AmplifyConv(channels, max_depth=32, num_module=1)
        
        self.max_pool = nn.AdaptiveMaxPool2d((1, 1))
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))


    def forward(self, x):
        
        x = self.conv0(x)

        x = self.amconv(x)

        x = torch.cat([self.max_pool(x), self.avg_pool(x)], dim=1)
        
        return x

In [7]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        
        self.reamconv= RecursionAmplifyConv(512, 32, 32)

        self.fc1 = nn.Linear(1024, 10)

    def forward(self, x):
        
        x = self.reamconv(x)
        
        x = x.contiguous().view(-1, 1024)
        x = F.relu(x)
        x = self.fc1(x)
        return x

In [8]:
def train_model(
    net: torch.nn.Module,
    get_cifar10_loaders,  # existing dataloader factory
    evaluate_model,       # existing eval helper
    batch_size: int = 64,
    epochs: int = 300,
    lr: float = 1e-3,
    eval_interval: int = 10,
    checkpoint_path: str = "Reasoning_32R_512C.pth",
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net = net.to(device)

    # data
    trainloader, testloader = get_cifar10_loaders(
        batch_size=batch_size,
        num_workers=8,
        pin_memory=True,
        persistent_workers=True,
    )

    # optimisation
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6
    )
    scaler = torch.amp.GradScaler(enabled=device.type == "cuda")

    best_test_acc = 0.0

    for epoch in range(epochs):
        net.train()
        running_loss = 0.0

        for inputs, targets in trainloader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # forward – mixed precision
            with torch.amp.autocast(device_type="cuda"):
                outputs = net(inputs)
                loss = criterion(outputs, targets)

            # backward + step
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=0.5)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        scheduler.step()

        avg_loss = running_loss / len(trainloader)

        print(f"Epoch [{epoch + 1}] loss: {avg_loss:.3f}")

        # evaluation & checkpointing
        if (epoch + 1) % eval_interval == 0 or epoch == epochs - 1:
            print(f"\nEvaluating at epoch {epoch + 1} …")
            train_acc = evaluate_model(net, trainloader, criterion, "Train", device)
            test_acc = evaluate_model(net, testloader, criterion, "Test", device)
            print(
                f"Current learning rate: {optimizer.param_groups[0]['lr']:.2e}\n"
            )

            if test_acc > best_test_acc:
                best_test_acc = test_acc
                torch.save(net.state_dict(), checkpoint_path)
                print(f"New best model saved with test accuracy: {test_acc:.2f}%")

    print("Finished Training")
    print(f"Best test accuracy achieved: {best_test_acc:.2f}%")

In [9]:
def evaluate_model(
    net: torch.nn.Module,
    dataloader,
    criterion,
    dataset_name: str = "",
    device=None,
):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    was_training = net.training
    net.eval()

    total_correct = 0
    total_seen = 0
    loss_sum = 0.0

    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    autocast_ctx = (torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16)
                    if use_bf16 else contextlib.nullcontext())

    with torch.no_grad(), autocast_ctx:
        for images, labels in dataloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = net(images)  
            batch_loss = criterion(outputs.float(), labels)

            if not torch.isfinite(batch_loss):
                print("[eval/warn] non-finite loss",
                      "logits_minmax=", float(outputs.min()), float(outputs.max()))
                continue

            bs = labels.size(0)
            loss_sum    += batch_loss.item() * bs
            total_seen  += bs

            preds = outputs.argmax(dim=1)
            total_correct += (preds == labels).sum().item()

    if total_seen == 0:
        avg_loss = float("nan")
        acc = 0.0
    else:
        avg_loss = loss_sum / total_seen
        acc = 100.0 * total_correct / total_seen

    print(f"{dataset_name} Accuracy: {acc:.2f}%")
    print(f"{dataset_name} Average Loss: {avg_loss:.4f}")

    if was_training:
        net.train()

    return acc

In [10]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Example usage:
model = Net()
print(f"Total Trainable Parameters: {count_parameters(model):,}")

Total Trainable Parameters: 515,754


In [11]:
torch.backends.cudnn.benchmark = True
net = Net()
train_model(net, get_cifar10_loaders, evaluate_model)

model = Net()
print(f"Total Trainable Parameters: {count_parameters(model):,}")

Epoch [1] loss: 1.879
Epoch [2] loss: 1.419
Epoch [3] loss: 1.228
Epoch [4] loss: 1.115
Epoch [5] loss: 1.023
Epoch [6] loss: 0.957
Epoch [7] loss: 0.909
Epoch [8] loss: 0.859
Epoch [9] loss: 0.831
Epoch [10] loss: 0.788

Evaluating at epoch 10 …
Train Accuracy: 73.59%
Train Average Loss: 0.7594
Test Accuracy: 78.68%
Test Average Loss: 0.6122
Current learning rate: 9.97e-04

New best model saved with test accuracy: 78.68%
Epoch [11] loss: 0.755
Epoch [12] loss: 0.738
Epoch [13] loss: 0.714
Epoch [14] loss: 0.694
Epoch [15] loss: 0.674
Epoch [16] loss: 0.656
Epoch [17] loss: 0.645
Epoch [18] loss: 0.635
Epoch [19] loss: 0.618
Epoch [20] loss: 0.604

Evaluating at epoch 20 …
Train Accuracy: 78.98%
Train Average Loss: 0.6047
Test Accuracy: 84.45%
Test Average Loss: 0.4497
Current learning rate: 9.89e-04

New best model saved with test accuracy: 84.45%
Epoch [21] loss: 0.596
Epoch [22] loss: 0.583
Epoch [23] loss: 0.568
Epoch [24] loss: 0.565
Epoch [25] loss: 0.558
Epoch [26] loss: 0.538
E